🧱 Bloque 1 – Importación de librerías

Este bloque reúne todas las herramientas necesarias para el proceso completo de automatización:
- Manejo de datos con pandas y numpy.
- Conexión a SQL Server mediante sqlalchemy.
- Manipulación de rutas, archivos y fechas con Path, datetime, time y shutil.
- Limpieza y normalización de textos con re y unicodedata.

Con estas librerías se prepara el entorno para leer, transformar y cargar los datos de forma segura.

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from datetime import datetime
from sqlalchemy import text
import sys
import importlib
import json
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/colmena/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path.cwd().resolve().parents[2]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR    = (PROJECT_ROOT / "config").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "colmena", "config_colmena.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🏗️ Bloque 2 – Configuración de conexión a la base de datos

Este bloque establece todos los parámetros necesarios para conectar con SQL Server:
- Servidor, base de datos, esquema y tabla destino.
- Creación del connection_string usando autenticación de Windows.
- Configuración del engine con fast_executemany para cargas rápidas y pool_pre_ping para mayor estabilidad.

A partir de aquí, cualquier ejecución SQL quedará lista para funcionar.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


🗂️ Bloque 3 – Configuración de archivos y hoja objetivo

Aquí se define dónde buscar los archivos Excel y qué hojas se consideran válidas:
- Ruta base del proceso.
- Extensiones permitidas.
- Lista de hojas preferidas y una hoja de respaldo.

Con esto, el proceso puede adaptarse automáticamente a diferentes archivos y formatos.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "colmena").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx",)  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["MaestroCesantiaPlus", "maestrocesantiaplus"]
FALLBACK_SHEET_INDEX = 0


✨ Bloque 4 – Normalización de nombres de columnas

Este bloque contiene funciones que limpian y estandarizan los nombres de columnas:
- Elimina acentos.
- Convierte a minúsculas.
- Reemplaza espacios y caracteres especiales.

Esto evita fallos cuando los archivos cambian encabezados o formatos.

📄 Bloque 5 – Detección de archivo más reciente y selección de hoja

En este bloque se realiza:
- Validación de que la carpeta exista.
- Selección del Excel más reciente.
- Manejo automático de bloqueo OneDrive mediante copia temporal.
- Búsqueda de la hoja objetivo entre las hojas disponibles.

De esta forma, el proceso siempre trabaja con la versión más actual del archivo.

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen  = info["archivo_origen"]
excel_path_used = info["excel_path_used"]
_tmp_copy_path  = info["tmp_copy_path"]
target          = info["target_sheet"]
sheet_names     = info["sheet_names"]

print(f"Archivo: {archivo_origen.name}")
print(f"Hoja seleccionada: {target}")


🧹 Bloque 6 – Lectura segura y limpieza inicial

Este bloque carga el Excel con lectura segura y prepara el DataFrame bruto:
    Intenta leer la hoja seleccionada y detiene el proceso con error claro si falla.
    Limpia textos vacíos o marcados como “None”.
    Normaliza los encabezados usando la función del bloque previo.

Deja el DataFrame listo para comenzar el mapeo de columnas.

In [25]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

if _tmp_copy_path is not None and _tmp_copy_path.exists():
    try:
        _tmp_copy_path.unlink()
        print(f"🗑️ Copia temporal eliminada: {_tmp_copy_path}")
    except Exception as e:
        print(f"⚠️ No se pudo borrar la copia temporal '{_tmp_copy_path}': {e}")

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
df_raw.head()

Encabezados normalizados: ['rut_titular', 'dv', 'nombre_titular', 'id_producto', 'producto', 'costo_total', 'fecha_nacimiento', 'fecha_suscripcion', 'monto_prima']


,rut_titular,dv,nombre_titular,id_producto,producto,costo_total,fecha_nacimiento,fecha_suscripcion,monto_prima
0,22473698,3,DOUGLAS ARTURO ROJAS GUISADO,69,Cesantía Plus 4 meses,12.1120,1991-05-14 00:00:00,2025-11-21 00:00:00,0.118
1,19632362,7,MARITZA ANDREA HERRERA PENALOZA,69,Cesantía Plus 4 meses,3.4300,1997-03-16 00:00:00,2025-11-23 00:00:00,0.118
2,18736959,2,PEDRO MOISES NAVARRO PEREZ,69,Cesantía Plus 4 meses,5.9500,1993-12-31 00:00:00,2024-11-20 00:00:00,0.118
3,18898746,K,CLAUDIO ANDRES PUEBLA SAUNERO,69,Cesantía Plus 4 meses,3.5440,1995-01-03 00:00:00,2025-11-24 00:00:00,0.118
4,17482956,K,DAVID ALFREDO VERA CESPED,69,Cesantía Plus 4 meses,4.1840,1990-03-30 00:00:00,2024-08-05 00:00:00,0.118


🎯 Bloque 7 – Selección de columnas desde múltiples posibles nombres

Este bloque busca las columnas correctas aunque los encabezados cambien en cada archivo:
- La función pick() permite buscar entre varios nombres posibles.
- Se toma la versión encontrada y, si no existe, se usa una columna con valores nulos.
- Se arma el DataFrame final con las columnas base.

Esto hace el proceso robusto frente a cambios externos.

In [26]:
def pick(df, *names, required=False, default=None):
    for n in names:
        if n in df.columns:
            return df[n]
    if required:
        raise KeyError(f"No se encontró ninguna de las columnas: {names}")
    return pd.Series([default] * len(df), index=df.index)


# Origen → Destino
guia                = pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
rut                 = pick(df_raw, "afirut", "rut_afiliado", "rut", "rutafiliado", "rut_titular")
dv                  = pick(df_raw, "afirutdv", "dv_afiliado", "dv", "dvafiliado")
id_producto         = pick(df_raw, "id_producto", "idproducto", "producto_id", "productoid")
producto            = pick(df_raw, "producto","nombre_producto", "nombreproducto")
costo_total         = pick(df_raw, "costo_total", "costototal", "monto_total", "montototal")
fecha_nacimiento    = pick(df_raw, "fecha_nacimiento", "fechanacimiento", "fec_nacimiento", "fecnacimiento")
fecha_suscripcion   = pick(df_raw, "fecha_suscripcion", "fechasuscripcion", "fec_suscripcion", "fecsuscripcion")
prima_uf            = pick(df_raw, "monto_prima", "montoprima", "prima_uf", "primauf")
poliza              = pick(df_raw, "fecotorgamiento", "fecha_otorgamiento", "fec_otorgamiento")




# Construimos DF destino (aún sin tipar)
df_can = pd.DataFrame({
    "Guia": guia,
    "RUT": rut,
    "DV": dv,
    "ID_Producto": id_producto,
    "Producto": producto,
    "Costo_Total": costo_total,
    "Fecha_Nacimiento": fecha_nacimiento,
    "Fecha_Suscripcion": fecha_suscripcion,
    "Prima_UF": prima_uf,
    "NOMBRE_ARCHIVO": archivo_origen.name,
})

df_can.head()

,Guia,RUT,DV,ID_Producto,Producto,Costo_Total,Fecha_Nacimiento,Fecha_Suscripcion,Prima_UF,NOMBRE_ARCHIVO
0,None,22473698,3,69,Cesantía Plus 4 meses,12.1120,1991-05-14 00:00:00,2025-11-21 00:00:00,0.118,202512Full MaestroCesantiaSura.xlsx
1,None,19632362,7,69,Cesantía Plus 4 meses,3.4300,1997-03-16 00:00:00,2025-11-23 00:00:00,0.118,202512Full MaestroCesantiaSura.xlsx
2,None,18736959,2,69,Cesantía Plus 4 meses,5.9500,1993-12-31 00:00:00,2024-11-20 00:00:00,0.118,202512Full MaestroCesantiaSura.xlsx
3,None,18898746,K,69,Cesantía Plus 4 meses,3.5440,1995-01-03 00:00:00,2025-11-24 00:00:00,0.118,202512Full MaestroCesantiaSura.xlsx
4,None,17482956,K,69,Cesantía Plus 4 meses,4.1840,1990-03-30 00:00:00,2024-08-05 00:00:00,0.118,202512Full MaestroCesantiaSura.xlsx


🔄 Bloque 8 – Asignación de pólizas según ID_Producto

Este bloque aplica un diccionario de mapeo:
- Convierte cada ID_Producto en su correspondiente número de póliza.
- Muestra cuántas filas recibió cada póliza.
- Identifica si existen ID_Producto sin mapeo.

Garantiza que cada fila tenga su póliza correctamente asociada.

In [27]:
 # Diccionario de mapeo ID_Producto → Poliza
MAPA_POLIZAS = {
    "69":	7391284,
    "70":	7391285,
    "71":	7391286,
}


df_can["Poliza"] = df_can["ID_Producto"].map(MAPA_POLIZAS)

print("📊 Cantidad de filas por Poliza:")
print(df_can["Poliza"].value_counts(dropna=False))

faltantes = df_can[df_can["Poliza"].isna()]["ID_Producto"].unique()
if len(faltantes) > 0:
    print("⚠️ ID_Producto sin Poliza definido en el mapa:", faltantes)

📊 Cantidad de filas por Poliza:
Poliza
7391285    2010
7391286    1705
7391284    1697
Name: count, dtype: int64


🧭 Bloque 9 – Estandarización del nombre del archivo (canónico)

Este bloque:
- Detecta automáticamente el período YYYYMM del nombre del archivo.
- Corrige variaciones como abreviaturas, tildes o diferentes formatos de fecha.
- Genera un nombre estandarizado para evitar duplicados.
- Actualiza la columna NOMBRE_ARCHIVO con el nombre canónico final.

Garantiza consistencia y evita errores al cargar datos repetidos.

In [28]:
def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}


def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = _strip_accents(stem).upper()

    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        for mes_txt, mm in MESES.items():
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")


def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado: 'SC_MARSHYYYYMM.xlsx'
    Si no logra extraer el período (YYYYMM), devuelve un nombre genérico con aviso.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"{yyyymm} Cesantia Plus.xlsx"
    except ValueError as e:
        print(f"⚠️ No se pudo extraer período desde el nombre '{nombre}'. "
              f"Se usará un nombre genérico. Detalle: {e}")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"{timestamp} Cesantia Plus.xlsx"


nombre_canonico = canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)


df_can["NOMBRE_ARCHIVO"] = nombre_canonico

Nombre original : 202512Full MaestroCesantiaSura.xlsx
Nombre canónico : 202512 Cesantia Plus.xlsx


📆 Bloque 10 – Extracción automática del período (YYYYMM)

Aquí se obtiene el período directamente desde el nombre del archivo:
- Se extrae el año y mes usando una expresión regular.
- Se genera la columna MES_RECAUDACION.
- Se advierte si alguna fila no pudo extraer el período.

De esta forma, cada carga queda etiquetada con su mes de recaudación.

In [29]:
def extraer_yyyymm_de_nombre_archivo(df: pd.DataFrame, nombre_columna: str = "NOMBRE_ARCHIVO") -> pd.Series:
    return df[nombre_columna].str.extract(r"(20\d{2})(0[1-9]|1[0-2])", expand=False).apply(lambda x: ''.join(x), axis=1)

df_can["MES_RECAUDACION"] = extraer_yyyymm_de_nombre_archivo(df_can)

if df_can["MES_RECAUDACION"].isnull().any():
    print("⚠️ Algunos valores de MES_RECAUDACION no pudieron ser extraídos.")
    
df_can[["NOMBRE_ARCHIVO", "MES_RECAUDACION"]].head(3)

,NOMBRE_ARCHIVO,MES_RECAUDACION
0,202512 Cesantia Plus.xlsx,202512
1,202512 Cesantia Plus.xlsx,202512
2,202512 Cesantia Plus.xlsx,202512


🗓️ Bloque 11 – Formateo seguro de fechas y creación YYYYMMDD

Este bloque convierte fechas y genera nuevas columnas:
- Convierte Fecha_Nacimiento y Fecha_Suscripcion a datetime seguro.
- Crea las columnas Fecha_Nac y Fecha_Sus en formato YYYYMMDD.
- Muestra cuántas filas fueron convertidas correctamente y cuántas quedaron nulas.

Establece las fechas en un formato consistente para SQL Server.

In [30]:
df_can["Fecha_Nacimiento"]  = pd.to_datetime(df_can["Fecha_Nacimiento"],  errors="coerce")
df_can["Fecha_Suscripcion"] = pd.to_datetime(df_can["Fecha_Suscripcion"], errors="coerce")

nac_null_before = df_can["Fecha_Nacimiento"].isna().sum()
sus_null_before = df_can["Fecha_Suscripcion"].isna().sum()

df_can["Fecha_Nac"] = df_can["Fecha_Nacimiento"].dt.strftime("%Y%m%d")
df_can["Fecha_Sus"] = df_can["Fecha_Suscripcion"].dt.strftime("%Y%m%d")

nac_null_after = df_can["Fecha_Nac"].isna().sum()
sus_null_after = df_can["Fecha_Sus"].isna().sum()

print("=== RESUMEN CREACIÓN FECHAS ===")

print(f"Fecha_Nacimiento:")
print(f"  - Nulos antes de convertir: {nac_null_before}")
print(f"  - Nulos en Fecha_Nac (formato YYYYMMDD): {nac_null_after}")
print(f"  - Filas convertidas correctamente: {len(df_can) - nac_null_after}")

print(f"\nFecha_Suscripcion:")
print(f"  - Nulos antes de convertir: {sus_null_before}")
print(f"  - Nulos en Fecha_Sus: {sus_null_after}")
print(f"  - Filas convertidas correctamente: {len(df_can) - sus_null_after}")

=== RESUMEN CREACIÓN FECHAS ===
Fecha_Nacimiento:
  - Nulos antes de convertir: 0
  - Nulos en Fecha_Nac (formato YYYYMMDD): 0
  - Filas convertidas correctamente: 5412

Fecha_Suscripcion:
  - Nulos antes de convertir: 0
  - Nulos en Fecha_Sus: 0
  - Filas convertidas correctamente: 5412


🛠️ Bloque 12 – Tipificación completa para compatibilidad SQL

Este Bloque se lleva cada columna al tipo correcto:
- Enteros → Int64.
- Decimales → float64.
- Fechas → tipo date puro.
- Textos → cadenas con longitud limitada según tabla.
- Validación de nulos en columnas críticas.
- Creación del DataFrame final df_sql con el orden exacto requerido.

Esto asegura que la carga no tenga errores de tipo.

In [31]:
INT_COLS    = ["Guia", "RUT", "DV", "ID_Producto",  "Poliza", "MES_RECAUDACION", "Fecha_Nac", "Fecha_Sus"]
FLOAT_COLS  = ["Costo_Total","Prima_UF"]
STR_COLS    = ["Producto", "NOMBRE_ARCHIVO"]
DATE_COLS   = ["Fecha_Nacimiento", "Fecha_Suscripcion"]

def _to_num_series(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip().replace({
        "": np.nan,
        "None": np.nan,
        "none": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
    })
    return pd.to_numeric(s2, errors="coerce")


def convert_to_datetime(df: pd.DataFrame, date_columns: list, *, dayfirst: bool = False) -> pd.DataFrame:
    """
    Convierte las columnas de fechas a tipo date (YYYY-MM-DD sin hora).
    - Si la conversión falla, reemplaza con NaT/None.
    - Deja el valor final como datetime.date, ideal para columnas DATE en SQL Server.
    """
    for col in date_columns:
        if col in df.columns:
            dt = pd.to_datetime(
                df[col],
                errors="coerce",
                dayfirst=dayfirst,
                infer_datetime_format=True,
            )
            df[col] = dt.dt.date

    return df

df_can = convert_to_datetime(df_can, DATE_COLS)


# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        s = _to_num_series(df_can[c])

        frac_mask = (s.notna()) & (s % 1 != 0)
        if frac_mask.any():
            print(f"⚠️ Columna '{c}' (INT) tiene {frac_mask.sum()} valores con decimales. Se redondearán antes de convertir a Int64.")
            # Aproximar al entero más cercano
            s = s.round(0)

        df_can[c] = s.astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")


if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 300)

if "NOMBRE_ARCHIVO" in df_can.columns:
    df_can["NOMBRE_ARCHIVO"] = df_can["NOMBRE_ARCHIVO"].astype("string").str.strip().str.slice(0, 255)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

criticas = ["RUT","Poliza","NOMBRE_ARCHIVO"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "Guia", "RUT", "DV", "ID_Producto", "Producto", "Costo_Total", "Fecha_Nacimiento", 
    "Fecha_Suscripcion", "Prima_UF", "Poliza", "NOMBRE_ARCHIVO", "MES_RECAUDACION",
    "Fecha_Nac", "Fecha_Sus", 
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_16304\967691562.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_16304\967691562.py:25: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_16304\967691562.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behav

✅ dtypes DESPUÉS:
 Guia                          Int64
RUT                           Int64
DV                            Int64
ID_Producto                   Int64
Producto             string[python]
Costo_Total                 float64
Fecha_Nacimiento             object
Fecha_Suscripcion            object
Prima_UF                    float64
NOMBRE_ARCHIVO       string[python]
Poliza                        Int64
MES_RECAUDACION               Int64
Fecha_Nac                     Int64
Fecha_Sus                     Int64
dtype: object

🔎 Nulos en columnas críticas:
 - RUT: 0 nulos
 - Poliza: 0 nulos
 - NOMBRE_ARCHIVO: 0 nulos

📦 df_sql listo con columnas: ['Guia', 'RUT', 'DV', 'ID_Producto', 'Producto', 'Costo_Total', 'Fecha_Nacimiento', 'Fecha_Suscripcion', 'Prima_UF', 'Poliza', 'NOMBRE_ARCHIVO', 'MES_RECAUDACION', 'Fecha_Nac', 'Fecha_Sus']


🛑 Bloque 12 – Verificación de NOMBRE_ARCHIVO y Control de Cargas Previas

Este bloque garantiza que solo se carguen archivos válidos y que la carga sea segura:

- Verifica que la columna NOMBRE_ARCHIVO exista en el DataFrame.
- Normaliza sus valores (quita espacios, controla vacíos).
- Identifica todos los valores únicos (útil para consolidados con varios meses).
- Muestra un conteo por cada archivo detectado.

Prepara la variable vals, que será usada para decidir:

- Si cada archivo será insertado, o si será reemplazado en SQL Server.

In [32]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 NOMBRE_ARCHIVO distintos en el df_sql:
   - 202512 Cesantia Plus.xlsx: 5412 filas


🚀 Bloque 13 – Carga Dinámica a SQL Server + Reemplazo Inteligente

Este bloque realiza la carga final al Data Warehouse de forma controlada y segura.

Para cada NOMBRE_ARCHIVO detectado:

- Verifica si ya existe en SQL Server.    
- Si existe → elimina esas filas.   
- Inserta los datos nuevos del archivo (o consolidado).   
- Todo dentro de una sola transacción, evitando inconsistencias.    
- Genera un resumen detallado por archivo: reemplazo o inserción nueva.

In [33]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals:  
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")


🆕 No hay data previa para NOMBRE_ARCHIVO = '202512 Cesantia Plus.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 5412 filas para NOMBRE_ARCHIVO = '202512 Cesantia Plus.xlsx'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - 202512 Cesantia Plus.xlsx: 5412 filas cargadas (archivo nuevo).


🗑️ Bloque 14 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [34]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\Colmena\202512Full MaestroCesantiaSura.xlsx
